# Train the Trashy detector (D-FINE nano)

This notebook trains a real object detector for Trashy: it finds every item in a photo, draws a box around it, and sorts it into **Paper, Plastic, Glass, Metal or Trash**.

**How to run it**
1. Menu **Runtime → Change runtime type → T4 GPU** → Save.
2. Menu **Runtime → Run all**.
3. Wait. Downloading data takes ~10 minutes, training ~1–2 hours. Keep this tab open.
4. At the end your browser downloads **trashy-detector.zip**. Send it to Claude (or put it in the Trashy folder as described in the last cell).

Everything runs on Google's computers. Training data comes from four open datasets (see the credits cell).

In [ ]:
#@title 1. Set up (about 1 minute)
!nvidia-smi -L
!pip -q install "transformers>=4.53" datasets "albumentations>=1.4" pycocotools "torchmetrics>=1.4" onnx onnxruntime
import torch, transformers, albumentations
print("torch", torch.__version__, "| transformers", transformers.__version__, "| albumentations", albumentations.__version__)
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all"

In [ ]:
#@title 2. Settings
MODEL_ID = "ustc-community/dfine-nano-coco"   # bigger and more accurate (but slower in the browser): "ustc-community/dfine-small-coco"
EPOCHS   = 40
BATCH    = 16
LR       = 1e-4
IMG      = 640
CLASSES  = ["Paper", "Plastic", "Glass", "Metal", "Trash"]
C = {n: i for i, n in enumerate(CLASSES)}
WORK = "/content/trashy"
import os; os.makedirs(f"{WORK}/img", exist_ok=True)

## Training data
- **TrashNet** (Gary Thung & Mindy Yang, MIT): single items on a white board; boxes are found automatically.
- **dmedhi garbage dataset** (CC BY 4.0): items with boxes. Its validation split is used for testing.
- **Bower waste annotations** (MIT): real homes. 25% of the photos are kept for testing (same split as Trashy's earlier tests).
- **TACO** (CC BY 4.0, photos from Flickr): litter in the wild. 15% of the photos are kept for testing.

In [ ]:
#@title 3. Download and prepare the data (about 10 minutes)
import json, io, zipfile, random, urllib.request, concurrent.futures as cf
import numpy as np
from PIL import Image, ImageOps
from datasets import load_dataset
from huggingface_hub import hf_hub_download

def fnv(s):                       # same split hash as Trashy's earlier tests
    h = 2166136261
    for ch in s:
        h = ((h ^ ord(ch)) * 16777619) & 0xFFFFFFFF
    return h / 4294967296

records = []                      # {"file", "boxes":[[x,y,w,h] px], "labels":[int], "split", "ds"}
def save(img, name, boxes, labels, split, ds, max_side=1024):
    img = ImageOps.exif_transpose(img).convert("RGB")
    s = min(1.0, max_side / max(img.size))
    if s < 1:
        img = img.resize((round(img.width * s), round(img.height * s)), Image.BILINEAR)
        boxes = [[v * s for v in b] for b in boxes]
    W, H = img.size
    keep_b, keep_l = [], []
    for (x, y, w, h), l in zip(boxes, labels):
        x, y = max(0, x), max(0, y); w, h = min(W - x, w), min(H - y, h)
        if w > 4 and h > 4:
            keep_b.append([x, y, w, h]); keep_l.append(l)
    if not keep_b: return
    path = f"{WORK}/img/{name}.jpg"; img.save(path, quality=92)
    records.append({"file": path, "boxes": keep_b, "labels": keep_l, "split": split, "ds": ds})

# --- dmedhi (boxes are x, y, w, h in pixels) ---
DM = {"Cardboard": "Paper", "Paper": "Paper", "Glass": "Glass", "Metal": "Metal", "Plastic": "Plastic", "Trash": "Trash", "Garbage": "Trash"}
dm = load_dataset("dmedhi/garbage-image-classification-detection")
for split, name in [("train", "train"), ("validation", "test")]:
    for i, r in enumerate(dm[split]):
        c = DM.get(r["class_name"])
        if c and r["bbox"]:
            save(r["image"], f"dm-{split}-{i}", r["bbox"], [C[c]] * len(r["bbox"]), name, "dmedhi")
print("dmedhi done", len(records))

# --- Bower (one row per box; x1, y1, x2, y2 in pixels) ---
BW = {"Fiber - Cardboard / Paper Packaging": "Paper", "Fiber - Paper": "Paper", "Metal": "Metal", "Plastic - Hard": "Plastic",
      "Plastic - Soft": "Plastic", "Glass - Non-coloured": "Glass", "Glass - Coloured": "Glass"}
bw = load_dataset("BowerApp/bower-waste-annotations")["train"]
by = {}
for r in bw: by.setdefault(r["image_id"], []).append(r)
for iid, rows in by.items():
    b, l = [], []
    for r in rows:
        c = BW.get(r["material"])
        if c: b.append([r["x1"], r["y1"], r["x2"] - r["x1"], r["y2"] - r["y1"]]); l.append(C[c])
    if b: save(rows[0]["image"], f"bw-{iid}", b, l, "test" if fnv(iid) < 0.25 else "train", "bower")
print("bower done", len(records))

# --- TACO (COCO format, photos on Flickr) ---
TACO = {"Metal": [0, 2, 8, 10, 11, 12, 28, 50, 52], "Glass": [6, 9, 23, 26],
        "Paper": [13, 14, 15, 17, 18, 19, 30, 32, 33, 34],
        "Plastic": [4, 5, 7, 21, 22, 24, 27, 29, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 49, 54, 55, 57],
        "Trash": [20, 25, 31, 48, 51, 58, 59]}
TMAP = {cid: C[k] for k, ids in TACO.items() for cid in ids}
ann = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json"))
boxes_by_img = {}
for a in ann["annotations"]:
    if a["category_id"] in TMAP: boxes_by_img.setdefault(a["image_id"], []).append((a["bbox"], TMAP[a["category_id"]]))
def taco_one(im):
    if im["id"] not in boxes_by_img: return None
    for url in [im.get("flickr_640_url"), im.get("flickr_url")]:
        if not url: continue
        try:
            data = urllib.request.urlopen(url, timeout=30).read()
            return im, Image.open(io.BytesIO(data))
        except Exception: pass
    return None
with cf.ThreadPoolExecutor(16) as ex:
    for res in ex.map(taco_one, ann["images"]):
        if not res: continue
        im, img = res
        img = ImageOps.exif_transpose(img)
        sx, sy = img.width / im["width"], img.height / im["height"]
        if abs(sx - sy) > 0.05:   # photo stored rotated compared with its boxes: skip it
            continue
        bl = boxes_by_img[im["id"]]
        save(img, f"taco-{im['id']}", [[x * sx, y * sy, w * sx, h * sy] for (x, y, w, h), _ in bl], [c for _, c in bl],
             "test" if fnv(im["file_name"]) < 0.15 else "train", "taco")
print("taco done", len(records))

# --- TrashNet (single items on a white board; box = everything that differs from the board) ---
TN = {"cardboard": "Paper", "paper": "Paper", "plastic": "Plastic", "glass": "Glass", "metal": "Metal", "trash": "Trash"}
z = zipfile.ZipFile(hf_hub_download("garythung/trashnet", "dataset-resized.zip", repo_type="dataset"))
for n in z.namelist():
    parts = n.split("/")
    if len(parts) != 3 or not n.endswith(".jpg") or parts[0].startswith("__"): continue
    img = Image.open(io.BytesIO(z.read(n))).convert("RGB"); a = np.asarray(img).astype(np.int16)
    border = np.concatenate([a[:8].reshape(-1, 3), a[-8:].reshape(-1, 3), a[:, :8].reshape(-1, 3), a[:, -8:].reshape(-1, 3)])
    diff = np.abs(a - np.median(border, 0)).max(2) > 45
    ys, xs = np.where(diff)
    if len(xs) < 500: continue
    x0, x1 = np.percentile(xs, [1, 99]); y0, y1 = np.percentile(ys, [1, 99])
    if (x1 - x0) * (y1 - y0) < 0.04 * a.shape[0] * a.shape[1]: continue
    save(img, f"tn-{parts[1]}-{parts[2][:-4]}", [[x0, y0, x1 - x0, y1 - y0]], [C[TN[parts[1]]]], "test" if fnv(parts[2]) < 0.15 else "train", "trashnet")
print("trashnet done", len(records))

import collections
cnt = collections.Counter((r["ds"], r["split"]) for r in records)
for k, v in sorted(cnt.items()): print(k, v, "photos")
print("boxes per class:", collections.Counter(CLASSES[l] for r in records if r["split"] == "train" for l in r["labels"]))
json.dump(records, open(f"{WORK}/records.json", "w"))

In [ ]:
#@title 4. Train (about 1-2 hours on a T4)
import albumentations as A
from transformers import AutoImageProcessor, AutoModelForObjectDetection, TrainingArguments, Trainer

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
train_recs = [r for r in records if r["split"] == "train"]
test_recs  = [r for r in records if r["split"] == "test"]

aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.7, 1.25), translate_percent=(-0.1, 0.1), rotate=(-12, 12), p=0.6),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.3),
    A.MotionBlur(p=0.1),
], bbox_params=A.BboxParams(format="coco", label_fields=["labels"], min_visibility=0.3, clip=True))

class DS(torch.utils.data.Dataset):
    def __init__(self, recs, train): self.recs, self.train = recs, train
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]; img = np.asarray(Image.open(r["file"]).convert("RGB"))
        boxes, labels = r["boxes"], r["labels"]
        if self.train:
            t = aug(image=img, bboxes=boxes, labels=labels); img, boxes, labels = t["image"], t["bboxes"], t["labels"]
        anns = [{"bbox": list(map(float, b)), "category_id": int(l), "area": float(b[2] * b[3]), "iscrowd": 0}
                for b, l in zip(boxes, labels) if b[2] > 1 and b[3] > 1]
        out = processor(images=img, annotations={"image_id": i, "annotations": anns}, return_tensors="pt")
        return {"pixel_values": out["pixel_values"][0], "labels": out["labels"][0]}

def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]), "labels": [b["labels"] for b in batch]}

model = AutoModelForObjectDetection.from_pretrained(
    MODEL_ID, id2label=dict(enumerate(CLASSES)), label2id=C, ignore_mismatched_sizes=True)

args = TrainingArguments(
    output_dir=f"{WORK}/run", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
    learning_rate=LR, weight_decay=1e-4, warmup_steps=int(0.05 * EPOCHS * len(train_recs) / BATCH), lr_scheduler_type="cosine", max_grad_norm=0.1,
    dataloader_num_workers=2, remove_unused_columns=False, save_strategy="epoch", save_total_limit=1,
    logging_steps=50, report_to="none", fp16=False)
trainer = Trainer(model=model, args=args, train_dataset=DS(train_recs, True), data_collator=collate)
trainer.train()
model.save_pretrained(f"{WORK}/final"); processor.save_pretrained(f"{WORK}/final")

In [ ]:
#@title 5. Test on photos it has never seen
from torchmetrics.detection import MeanAveragePrecision
import collections
model.eval().cuda()

def boxes_xyxy(b): return [[x, y, x + w, y + h] for x, y, w, h in b]
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)

report = {}
for ds in sorted(set(r["ds"] for r in test_recs)):
    recs = [r for r in test_recs if r["ds"] == ds]
    mapm = MeanAveragePrecision(iou_type="bbox"); found = right = total = 0
    per = collections.defaultdict(lambda: [0, 0])
    for k in range(0, len(recs), 16):
        chunk = recs[k:k + 16]; imgs = [Image.open(r["file"]).convert("RGB") for r in chunk]
        with torch.no_grad():
            out = model(**processor(images=imgs, return_tensors="pt").to("cuda"))
        res = processor.post_process_object_detection(out, threshold=0.05, target_sizes=[(im.height, im.width) for im in imgs])
        for r, p in zip(chunk, res):
            gt = boxes_xyxy(r["boxes"])
            mapm.update([{k2: v.cpu() for k2, v in p.items()}], [{"boxes": torch.tensor(gt, dtype=torch.float32), "labels": torch.tensor(r["labels"])}])
            pb, ps, pl = p["boxes"].tolist(), p["scores"].tolist(), p["labels"].tolist()
            for g, gl in zip(gt, r["labels"]):
                total += 1; per[CLASSES[gl]][1] += 1
                cand = [(s, l) for b, s, l in zip(pb, ps, pl) if s >= 0.35 and iou(b, g) >= 0.5]
                if cand:
                    found += 1; s, l = max(cand)
                    if l == gl: right += 1; per[CLASSES[gl]][0] += 1
    m = mapm.compute()
    report[ds] = {"mAP50": round(float(m["map_50"]), 3), "mAP": round(float(m["map"]), 3),
                  "items_found": round(found / total, 3), "items_right": round(right / total, 3),
                  "per_class_right": {c: round(a / b, 3) for c, (a, b) in per.items()}}
    print(f"{ds:9s} mAP50 {report[ds]['mAP50']:.3f} | items found {found/total:.1%} | found AND right category {right/total:.1%} |",
          ", ".join(f"{c} {a/b:.0%}" for c, (a, b) in per.items()))
json.dump(report, open(f"{WORK}/metrics.json", "w"), indent=1)

In [ ]:
#@title 6. Export for the browser (ONNX) and check it
import onnxruntime as ort
class Wrap(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, pixel_values):
        o = self.m(pixel_values=pixel_values); return o.logits, o.pred_boxes
cpu_model = model.float().cpu().eval()
x = processor(images=Image.open(test_recs[0]["file"]).convert("RGB"), return_tensors="pt")["pixel_values"]
torch.onnx.export(Wrap(cpu_model), x, f"{WORK}/model.onnx", input_names=["pixel_values"], output_names=["logits", "pred_boxes"],
                  opset_version=17, dynamo=False)
with torch.no_grad(): ref = Wrap(cpu_model)(x)
s = ort.InferenceSession(f"{WORK}/model.onnx", providers=["CPUExecutionProvider"])
o = s.run(None, {"pixel_values": x.numpy()})
print("ONNX matches PyTorch:", float(np.abs(o[0] - ref[0].numpy()).max()), float(np.abs(o[1] - ref[1].numpy()).max()))
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(f"{WORK}/model.onnx", f"{WORK}/model-int8.onnx", weight_type=QuantType.QUInt8)
for f in ["model.onnx", "model-int8.onnx"]: print(f, round(os.path.getsize(f"{WORK}/{f}") / 1e6, 1), "MB")

# how much does the small int8 file lose? (same 'found and right' test on 300 test photos, with ONNX Runtime)
def onnx_right(path, recs):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"]); right = total = 0
    for r in recs:
        im = Image.open(r["file"]).convert("RGB"); W, H = im.size
        lg, bx = s.run(None, {"pixel_values": processor(images=im, return_tensors="pt")["pixel_values"].numpy()})
        sc = 1 / (1 + np.exp(-lg[0])); q, l = np.unravel_index(np.argsort(-sc, axis=None)[:300], sc.shape)
        preds = [(sc[a, b], b, [(bx[0, a, 0] - bx[0, a, 2] / 2) * W, (bx[0, a, 1] - bx[0, a, 3] / 2) * H, (bx[0, a, 0] + bx[0, a, 2] / 2) * W, (bx[0, a, 1] + bx[0, a, 3] / 2) * H]) for a, b in zip(q, l) if sc[a, b] >= 0.35]
        for g, gl in zip(boxes_xyxy(r["boxes"]), r["labels"]):
            total += 1; cand = [(p, lab) for p, lab, b in preds if iou(b, g) >= 0.5]
            if cand and max(cand)[1] == gl: right += 1
    return right / total
sample = random.Random(1).sample(test_recs, min(300, len(test_recs)))
full, small = onnx_right(f"{WORK}/model.onnx", sample), onnx_right(f"{WORK}/model-int8.onnx", sample)
print(f"found and right: full {full:.1%} | int8 {small:.1%}")
best = "model-int8.onnx" if small >= full - 0.02 else "model.onnx"
print("the app will use", best)

In [ ]:
#@title 7. Download trashy-detector.zip
import shutil
from google.colab import files
out = f"{WORK}/trashy-detector"; os.makedirs(out, exist_ok=True)
shutil.copy(f"{WORK}/{best}", f"{out}/model.onnx")
json.dump({"format": "trashy-dfine-v1", "base": MODEL_ID, "classes": CLASSES, "input": IMG,
           "preprocess": "stretch to 640x640, RGB, divide by 255", "outputs": ["logits (sigmoid per class)", "pred_boxes (cx, cy, w, h, 0..1)"],
           "threshold": 0.35, "metrics": json.load(open(f"{WORK}/metrics.json"))}, open(f"{out}/detector.json", "w"), indent=1)
shutil.make_archive(f"{WORK}/trashy-detector", "zip", out)
files.download(f"{WORK}/trashy-detector.zip")
print("Done! Unzip trashy-detector.zip into Trashy-0.1/models/trashy-detector/ (or send it to Claude).")